<a href="https://colab.research.google.com/github/NeuromatchAcademy/course-content/blob/main/tutorials/W1D1_ModelTypes/student/W1D1_Intro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> &nbsp; <a href="https://kaggle.com/kernels/welcome?src=https://raw.githubusercontent.com/NeuromatchAcademy/course-content/main/tutorials/W1D1_ModelTypes/student/W1D1_Intro.ipynb" target="_parent"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" alt="Open in Kaggle"/></a>

# はじめに

## 概要

NMAの最初の2日間は、モデリングのプロセスとモデルとは何かについて学びます。NMAの残りの期間でさまざまなモデリングツールに取り組む前に、これらのメタモデリングの側面を理解することが重要です。したがって、今日はモデルの多様性と異なるモデルが私たちに何をもたらすかについて学びます。一般的に、誰もが異なる解析ツールが実験データから異なる種類の情報を抽出できるという直感を持っています。質問に応じて異なる解析を選びます。しかし、モデルについてはあまり意識されていません。しかし、モデルも同様で、異なる種類の質問に答えるために異なる種類のモデルを構築したいのです。すべてはあなたの目的次第です。そこで今日は、DayanとAbbott（2001）による分類に基づいて、「何を（what）」、「どのように（how）」、「なぜ（why）」の3種類のモデルを検討します。

各チュートリアルでは、同じデータ、すなわち神経細胞の活動電位間の時間間隔（インタースパイク間隔、ISI）を説明するためにこれらのモデルの一つを案内します。チュートリアル1では、ISI分布の形状を最もよく表す関数は何か（指数分布である）を問います。この「what」モデルはISI分布を簡潔に表現でき、例えばデータセット、課題条件、脳領域間でISIの特性を定量化することが可能です。チュートリアル2では、観察されたISI分布を生成するメカニズムは何かを問います。この「how」モデルは、システムが観察された挙動を生み出す具体的な方法を提案します。ここでは、興奮と抑制のバランスが指数分布のISIを生成することが示されます。最後に、なぜ指数分布が神経細胞における情報符号化の最適な方法であるのかを問います。「why」モデルは現象の根底にある原理を探ります。

研究では通常、記述的な「what」モデルから始めます。これらの例はモデルフィッティング、GLM、次元削減、深層学習の日に見られます。次に、メカニズムについて問い、「how」モデルを構築して基礎となるメカニズムの仮説を生成または検証します。これらの例は線形システム、実際の神経細胞、動的ネットワーク、意思決定の日にあります。最終的には、なぜその現象が存在するのかという根本的な理由に興味を持ちます。これらの例はベイズ、最適制御、強化学習の日に見られます。「why」モデルは達成が最も難しいことが多く、「what」モデルは通常最も簡単です。しかし、より重要なのは、それぞれが異なる質問に答え、異なる洞察を提供し、異なる有用性を持つことです。自分が答えたい質問、その質問に答えたい理由（すなわち目標）、評価したい仮説を考えることが、日々のモデリングの選択を決定します。結果として生まれるモデルの多様性は素晴らしく、すべてのモデルが問題の異なる側面に取り組んでいるため（今日の3つのチュートリアルのように）、知識探求において相補的です。今日の教材が、NMAで学ぶすべてのモデリングツールが提供する機会と限界をよりよく理解する助けになれば幸いです。

## 前提知識

今日の内容とチュートリアルでは、微分（W0D4 T1）、数値積分（オイラー法）（[W0D4 T3](https://precourse.neuromatch.io/tutorials/W0D4_Calculus/student/W0D4_Tutorial3.html)）、確率分布（[W0D5 T1](https://precourse.neuromatch.io/tutorials/W0D5_Statistics/student/W0D5_Tutorial1.html)）を使用します。また、神経細胞の基本知識（[Neuro Video Series Intro](https://precourse.neuromatch.io/tutorials/W0D0_NeuroVideoSeries/student/W0D0_Tutorial1.html)）も使用します。必要に応じてこれらのプレコース教材を復習してください！

In [ ]:
# @title Install and import feedback gadget


from vibecheck import DatatopsContentReviewContainer
def content_review(notebook_section: str):
    return DatatopsContentReviewContainer(
        "",  # No text prompt
        notebook_section,
        {
            "url": "https://pmyvdlilci.execute-api.us-east-1.amazonaws.com/klab",
            "name": "neuromatch_cn",
            "user_key": "y1x3mpx5",
        },
    ).render()


feedback_prefix = "W1D1_Intro"

## ビデオ

In [ ]:
# @markdown
from ipywidgets import widgets
from IPython.display import YouTubeVideo
from IPython.display import IFrame
from IPython.display import display


class PlayVideo(IFrame):
  def __init__(self, id, source, page=1, width=400, height=300, **kwargs):
    self.id = id
    if source == 'Bilibili':
      src = f'https://player.bilibili.com/player.html?bvid={id}&page={page}'
    elif source == 'Osf':
      src = f'https://mfr.ca-1.osf.io/render?url=https://osf.io/download/{id}/?direct%26mode=render'
    super(PlayVideo, self).__init__(src, width, height, **kwargs)


def display_videos(video_ids, W=400, H=300, fs=1):
  tab_contents = []
  for i, video_id in enumerate(video_ids):
    out = widgets.Output()
    with out:
      if video_ids[i][0] == 'Youtube':
        video = YouTubeVideo(id=video_ids[i][1], width=W,
                             height=H, fs=fs, rel=0)
        print(f'Video available at https://youtube.com/watch?v={video.id}')
      else:
        video = PlayVideo(id=video_ids[i][1], source=video_ids[i][0], width=W,
                          height=H, fs=fs, autoplay=False)
        if video_ids[i][0] == 'Bilibili':
          print(f'Video available at https://www.bilibili.com/video/{video.id}')
        elif video_ids[i][0] == 'Osf':
          print(f'Video available at https://osf.io/{video.id}')
      display(video)
    tab_contents.append(out)
  return tab_contents


video_ids = [('Youtube', 'W5o_HTsef0I'), ('Bilibili', 'BV1ho4y1C7Eo')]
tab_contents = display_videos(video_ids, W=854, H=480)
tabs = widgets.Tab()
tabs.children = tab_contents
for i in range(len(tab_contents)):
  tabs.set_title(i, video_ids[i][0])
display(tabs)

## スライド

In [ ]:
# @markdown
from IPython.display import IFrame
link_id = "rbx2a"
print(f"If you want to download the slides: https://osf.io/download/{link_id}/")
IFrame(src=f"https://mfr.ca-1.osf.io/render?url=https://osf.io/{link_id}/?direct%26mode=render%26action=download%26mode=render", width=854, height=480)

In [ ]:
# @title Submit your feedback
content_review(f"{feedback_prefix}_Intro_Video")